# Tesla T4 FNO training feasibility probe
One pseudo-epoch, 16 modes, 256 × 256, batch 64.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-fno-darcy.git"
REF = "e2faadffb561009c3a897f88f9ce835ceb74ac31"   # output of `git rev-parse HEAD` for the commit to run
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = "/tmp/physicsnemo-fno-darcy"

SESSION_BUDGET_SECONDS = 1200


In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert not dirty, "the working tree is not clean"
print("commit", head)

Install the pinned PhysicsNeMo commit and the CPU audit versions of PyTorch, Warp, NumPy and SciPy into a fresh virtual environment. The next check reports the installed versions and confirms CUDA availability.

In [ ]:
import sys
VENV = "/tmp/darcy-verification-env"
subprocess.run([sys.executable, "-m", "venv", "--without-pip", VENV], check=True)
PYTHON = f"{VENV}/bin/python"
subprocess.run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "-q", "pip"], check=True)
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
# Isolate from Kaggle's optional RAPIDS stack. Match the CPU audit's Warp and
# numerical libraries to make the cross-device comparison easier to interpret.
sh("python -m pip install -q -r requirements.txt torch==2.14.1 torchvision==0.29.1 "
   "warp-lang==1.18.0 numpy==2.5.3 scipy==1.18.1", cwd=REPO)


## Run the bounded probe and archive its evidence

In [ ]:
import json
import shutil
from pathlib import Path

evidence = Path(WORK) / "cuda_training_probe"
evidence.mkdir(exist_ok=False)
(evidence / "nvidia-smi.txt").write_text(subprocess.getoutput("nvidia-smi"))
(evidence / "pip-freeze.txt").write_text(subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True))
command = [PYTHON, "-u", "scripts/probe_cuda_fno_training.py", "--out", str(evidence)]
remaining = max(1, SESSION_BUDGET_SECONDS - (time.time() - T0))
with (evidence / "training.log").open("w") as log:
    try:
        status = subprocess.run(command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT,
                                timeout=remaining, text=True,
                                env={**os.environ, "MPLBACKEND": "Agg",
                                     "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"}).returncode
    except subprocess.TimeoutExpired:
        status = 124
print((evidence / "training.log").read_text())
report = evidence / "report.json"
if report.exists():
    result = json.loads(report.read_text())
else:
    result = {}
if status == 124:
    result.update(status="BLOCKED", reason="Session budget reached")
elif status:
    result.setdefault("status", "error")
result.update(notebook_elapsed_seconds=time.time() - T0, command_exit_status=status)
report.write_text(json.dumps(result, indent=2) + "\n")
archive = shutil.make_archive(f"{WORK}/cuda_training_probe", "zip", evidence)
print("Feasibility:", result.get("status"), "| optimizer updates:", result.get("optimizer_updates", 0))
print("Archive:", archive)
